# LGD: revolving-loan deep dive

Notebook version of `src/model/lgd_revolver_deepdive.py`: same population, same numbers, one question per cell, charts shown inline.

**Population:** every charged-off loan, no 60-month seasoning filter (a defaulted loan's loss is already known, whatever its age).

**The point of the whole notebook:** revolving loans are a different kind of product, not a footnote. Less collateral, less SBA guarantee, higher rate, faster default and much higher loss severity.

Nothing here writes to the repo. The saved charts (`outputs/figures/lgd/03_*` to `07_*`) are made by the script.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in [ROOT, ROOT / "src" / "features", ROOT / "src" / "model"]:
    sys.path.insert(0, str(p))

import pandas as pd
import matplotlib.pyplot as plt
from lgd_revolver_deepdive import build_population   # same function the script uses

# the script switches matplotlib to a no-display backend when imported; this switches it back
%matplotlib inline

pd.set_option("display.width", 200)

df = build_population()
print(f"{len(df):,} charged-off loans")

## 1. A loan's write-off often comes long after its scheduled term ends

For short loans (36 months or less): how often is the time to charge-off longer than the loan's own term? This is a timing pattern across the board, not a data error and not specific to revolvers.

In [ ]:
short = df[(df["TermInMonths"] > 0) & (df["TermInMonths"] <= 36)].copy()
short["charged off after term ended"] = short["MonthsToChargeoff"] > short["TermInMonths"]
out = short.groupby("RevolverStatus")["charged off after term ended"].agg(loans="size", pct=lambda s: s.mean() * 100)
out.loc["all"] = [len(short), short["charged off after term ended"].mean() * 100]
out.round(1)

## 2. How revolvers differ from non-revolvers

Share of loans with and without collateral, by revolver status (rows add to 100).

In [ ]:
(pd.crosstab(df["RevolverStatus"], df["CollateralInd"], normalize="index") * 100).round(1)

SBA guarantee share, by revolver status.

In [ ]:
df.groupby("RevolverStatus")["GuaranteePct"].agg(loans="size", mean="mean", median="median").round(2)

Months from approval to charge-off, by revolver status.

In [ ]:
df.groupby("RevolverStatus")["MonthsToChargeoff"].agg(loans="size", mean="mean", median="median").round(1)

Interest rate, by revolver status and collateral.

In [ ]:
df.groupby(["RevolverStatus", "CollateralInd"], observed=True)["InitialInterestRate"].agg(loans="size", mean="mean").round(2)

Realized loss severity, by revolver status and collateral.

In [ ]:
df.groupby(["RevolverStatus", "CollateralInd"], observed=True)["realized_lgd"].agg(loans="size", mean="mean").round(3)

## 3. Charts

### Loss severity by term and collateral

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
g = df.groupby(["term_bucket", "CollateralInd"], observed=True)["realized_lgd"].mean().unstack()
g.plot(kind="bar", ax=ax, color=["#DD8452", "#4C72B0"])
ax.set_ylabel("mean realized LGD")
ax.set_title("Realized LGD by term bucket x CollateralInd")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

### Loss severity by revolver status and collateral

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
g = df.groupby(["RevolverStatus", "CollateralInd"], observed=True)["realized_lgd"].mean().unstack()
g.plot(kind="bar", ax=ax, color=["#DD8452", "#4C72B0"])
ax.set_ylabel("mean realized LGD")
ax.set_title("Realized LGD by RevolverStatus x CollateralInd")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
plt.show()

### Months to charge-off by term and collateral

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
g = df.groupby(["term_bucket", "CollateralInd"], observed=True)["MonthsToChargeoff"].mean().unstack()
g.plot(kind="bar", ax=ax, color=["#DD8452", "#4C72B0"])
ax.set_ylabel("mean MonthsToChargeoff")
ax.set_title("MonthsToChargeoff by term bucket x CollateralInd")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

### SBA guarantee share, by revolver status and by collateral

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.boxplot(column="GuaranteePct", by="RevolverStatus", ax=axes[0])
axes[0].set_title("GuaranteePct by RevolverStatus")
axes[0].set_ylabel("GuaranteePct")
df.boxplot(column="GuaranteePct", by="CollateralInd", ax=axes[1])
axes[1].set_title("GuaranteePct by CollateralInd")
axes[1].set_ylabel("GuaranteePct")
fig.suptitle("")
fig.tight_layout()
plt.show()

### Interest rate and timing, by revolver status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
gm = df.groupby(["RevolverStatus", "CollateralInd"], observed=True)["InitialInterestRate"].mean().unstack()
gm.plot(kind="bar", ax=axes[0], color=["#DD8452", "#4C72B0"])
axes[0].set_title("Mean InitialInterestRate by RevolverStatus x CollateralInd")
axes[0].set_ylabel("mean InitialInterestRate")
axes[0].tick_params(axis="x", rotation=0)

gm2 = df.groupby("RevolverStatus")["MonthsToChargeoff"].mean()
axes[1].bar(gm2.index.astype(str), gm2.values, color="#55A868")
axes[1].set_title("Mean MonthsToChargeoff by RevolverStatus")
axes[1].set_ylabel("mean MonthsToChargeoff")
fig.tight_layout()
plt.show()